# Code-Graph Tier-1 — Binding-Precision Design Spec

> **Companion to** `2026-06-04-code-graph-ontology-tier0-design.ipynb`. Tier-0 settled the
> **T-Box** (what a class/predicate *is*, its domain/range, the realization contract). Tier-1
> is about the **A-Box made honest**: *given the schema is correct, does each edge point at the
> symbol the source code actually meant?* This is **binding precision** — disambiguating the
> latent AST so a resolved target is the real referent, not a same-named decoy.

Every claim below is backed by a read-only query against the live analyst artifact (the same
DuckDB the `code_*` MCP tools read). Evidence captured on artifact **`b322e436`** /
manifest **`a1f93dbbdf49`** — the post-v6 graph (singleton-calls crate-safety landed; the v7
references gate is committed but not yet rebuilt into this artifact, which is *why* you can still
see the 110 ungated `references` rows in §3).

**The one-sentence thesis of Tier-1:** *the graph has no model of any symbol outside the
worktree (stdlib, prelude, third-party crates), so a bare name with exactly one workspace
definition gets bound there — even when the real referent is external.* Every Tier-1 frontier is
a consequence of that **closed-world assumption**.


In [ ]:
# --- Live evidence harness: read-only connection to the SPUR analyst graph artifact ---
# Every "evidence" cell below queries this same DuckDB artifact the `code_*` MCP tools use.
import duckdb, pandas as pd

ANALYST_DB = "/Volumes/Projects/spur/.spur/analyst.duckdb"
con = duckdb.connect(ANALYST_DB, read_only=True)

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).fetchdf()

# Freshness + scale — Tier-1 reasoning is only valid against a known artifact.
q("""SELECT graph_content_hash[1:12] AS artifact, manifest_version[1:12] AS manifest,
            node_count, resolved_edge_count FROM _meta""")


     artifact      manifest  node_count  resolved_edge_count
0  b322e43647  a1f93dbbdf49       49989                94583


## 1. The closed-world root cause

The resolver binds an edge by **name matching against workspace symbols only**. There is no node
for `std::str::split_once`, `tokio::time::sleep`, or `serde_json::from_str` — the parser never
saw their definitions. So when a call site says `.split_once('=')` and the workspace happens to
contain exactly one symbol literally named `split_once`, the **singleton** path binds the call to
that lone workspace symbol. The edge is *resolved, high-confidence, and wrong*.

This single mechanism spawns **two opposite failure modes**, and Tier-1 is the work of telling
them apart:

| Failure mode | What happens | Tier-1 remedy |
|---|---|---|
| **Phantom (precision loss)** | A bare external name binds to a lone same-named workspace symbol across a crate boundary. | **Refuse the bind** unless provably same-scope. (v6/v7 ✅) |
| **Crater (recall loss)** | Refusing the bind drops *genuine* cross-crate workspace calls too — they go unresolved. | **Re-bind on evidence** (imports), not on name alone. (design, below) |

The predicate that draws the line is `function_singleton_safe` (`extract/tree_sitter.rs:1124`):
same-file → safe; else same `crates/<name>` scope → safe; cross-crate / non-crate → **unsafe**.


## 2. The bind-method landscape — where precision is won or lost

`bind_method` records *which resolution path* asserted each edge. It is the provenance axis Tier-1
reasons over: each path is an independent place a phantom can enter and an independent place a
guard must live.


In [ ]:
# Provenance of every resolved calls/references edge — the Tier-1 attack surface.
q("""SELECT relation, bind_method, count(*) AS n
     FROM edges WHERE relation IN ('calls','references')
     GROUP BY 1,2 ORDER BY 1, n DESC""")


    relation           bind_method     n
0      calls             singleton  9879
1      calls           scope_match  6462
2      calls                        2102
3      calls  macro_body_singleton  1280
4      calls                   fqn    21
5  references                        110


**Reading the table:**
- **`singleton`** (9,879) — lone-name bare-target binds. The primary phantom vector. Guarded by
  `function_singleton_safe` in `resolve_singleton_bare_target` (`tree_sitter.rs:845`) **and** in
  the rebind pass (`store/build.rs:1000`) since **v6**.
- **`scope_match`** (6,462) — method binds matched by enclosing-scope text. The **method frontier**
  (§5) lives here: 1,187 of these cross a crate boundary, and most are collisions.
- **`fqn`** (21) — qualified-path binds, callable-guarded since v5.
- **`references`** (110, all empty `bind_method`) — HOF function-value references. The **v7** gate
  (committed, not yet in this artifact) drops the cross-crate subset; this row is the *before*
  picture preserved in the captured artifact.


## 3. Frontier A — singleton phantoms ✅ (T1.b.2-a, landed v6 + v7)

The first and sharpest frontier: a singleton bare call/reference must not bind across a crate
boundary. **Done.** v6 gated `Calls`→function in both the resolver and the rebind pass; v7 mirrors
it for `References` (HOF). Evidence the calls half is live on this artifact:


In [ ]:
# Cross-crate singleton binds, by predicate. v6 drove calls cross-crate singletons to ZERO.
q("""
WITH x AS (
  SELECT e.bind_method,
         regexp_extract(src.file_path,'^crates/([^/]+)/',1) AS src_crate,
         regexp_extract(dst.file_path,'^crates/([^/]+)/',1) AS dst_crate
  FROM edges e
  JOIN nodes src ON src.stable_symbol_id=e.source_stable_id
  JOIN nodes dst ON dst.stable_symbol_id=e.target_stable_id
  WHERE e.relation='calls' AND e.bind_method IN ('singleton','scope_match')
)
SELECT bind_method,
       count(*) AS total,
       count(*) FILTER (WHERE src_crate<>'' AND dst_crate<>'' AND src_crate<>dst_crate) AS cross_crate
FROM x GROUP BY 1 ORDER BY 1
""")


   bind_method  total  cross_crate
0    scope_match   6461         1187
1      singleton   9788            0


`singleton` cross-crate = **0** — every former stdlib phantom (`.split_once` → lone workspace
`split_once`, etc.) is now left unresolved instead of mis-bound. `scope_match` still shows **1,187**
cross-crate — that is Frontier C (§5), untouched by v6/v7 by design.

> **Status:** ✅ Resolver + rebind gated, `RESOLVER_VERSION` at `references-crate-safety-v7`
> (`store/build.rs:29`). Regression-guarded by `singleton_hof_reference_respects_crate_safety`
> and `artifact_rebind_preserves_cross_crate_singleton_reference_unresolved`.


## 4. Frontier B — the recall crater ⚠️ (T1.b.2-b, the core Tier-1 design problem)

Refusing unsafe binds is honest, but it leaves a crater: **88,379 calls are unresolved.** Most are
genuinely external (std/third-party) and *should* stay unresolved — the graph cannot bind what it
never parsed. The recoverable subset is the calls whose bare label matches **exactly one** workspace
definition:


In [ ]:
# The recall pool: unresolved calls whose label matches a LONE workspace function/method,
# split by whether that lone def is in the caller's crate (safe) or another crate (needs proof).
q("""
WITH defs AS (
  SELECT entity_name, count(*) AS n,
         any_value(regexp_extract(file_path,'^crates/([^/]+)/',1)) AS def_crate
  FROM nodes WHERE symbol_kind IN ('function','method') GROUP BY entity_name
),
lone AS (SELECT entity_name, def_crate FROM defs WHERE n=1 AND def_crate<>''),
c AS (
  SELECT eu.target_label, regexp_extract(src.file_path,'^crates/([^/]+)/',1) AS call_crate
  FROM edges_unresolved eu
  JOIN nodes src ON src.stable_symbol_id = eu.source_stable_id
  WHERE eu.relation='calls'
)
SELECT
  (SELECT count(*) FROM edges_unresolved WHERE relation='calls') AS unresolved_calls_total,
  count(*)                                                        AS recall_pool_lone_match,
  count(*) FILTER (WHERE c.call_crate = l.def_crate)              AS same_crate_safe,
  count(*) FILTER (WHERE c.call_crate<>'' AND c.call_crate<>l.def_crate) AS cross_crate_needs_proof
FROM c JOIN lone l ON l.entity_name = c.target_label
""")


   unresolved_calls_total  recall_pool_lone_match  same_crate_safe  cross_crate_needs_proof
0                   88379                    7756             2810                     4920


**The crater splits into two recovery tiers with very different safety:**

- **2,810 same-crate, lone-def** — `function_singleton_safe` *already returns true* for these.
  They are unresolved only because the singleton path didn't fire (receiver-typed method position,
  or the label wasn't a global singleton at resolve time). **Recoverable with no new evidence** — a
  conservative same-crate recall pass. The cheapest, safest Tier-1 recall win.
- **4,920 cross-crate, lone-def** — binding these on name alone is *exactly the phantom v6 forbids*.
  They can only be recovered with **independent proof** that the call really targets that crate.
  The natural proof is a backing `import`/`use` edge in the same file. **But that proof is not yet
  trustworthy** — see §5.


## 5. Frontier C — imports can't license a bind until imports are themselves resolved ❌

The intuitive fix for the 4,920 is: *bind the cross-crate call only if the source file `use`s a path
ending in that name.* The problem: **import edges are bare-name unresolved too**, and import labels
**collide with workspace symbol names** — so naive import-name matching re-introduces the very
phantoms v6 removed.


In [ ]:
# The collision hazard: distinct import labels that EXACTLY equal a workspace function/method name.
q("""
WITH wfun AS (SELECT DISTINCT entity_name FROM nodes WHERE symbol_kind IN ('function','method'))
SELECT
  count(DISTINCT eu.target_label) AS distinct_import_labels,
  count(DISTINCT eu.target_label) FILTER (WHERE w.entity_name IS NOT NULL) AS collide_with_workspace_fn
FROM edges_unresolved eu
LEFT JOIN wfun w ON w.entity_name = eu.target_label
WHERE eu.relation='imports'
""")


   distinct_import_labels  collide_with_workspace_fn
0                    1346                         65


In [ ]:
# Concrete colliders — the import sites that would falsely bind under naive name-matching.
q("""
WITH wfun AS (SELECT DISTINCT entity_name,
                     regexp_extract(file_path,'^crates/([^/]+)/',1) AS crate
              FROM nodes WHERE symbol_kind IN ('function','method'))
SELECT eu.target_label, any_value(w.crate) AS workspace_def_crate, count(*) AS import_sites
FROM edges_unresolved eu
JOIN wfun w ON w.entity_name = eu.target_label
WHERE eu.relation='imports'
GROUP BY 1 ORDER BY import_sites DESC LIMIT 10
""")


    target_label workspace_def_crate  import_sites
0            fmt            spur-mcp          121
1           path             spur-pm           84
2         render           spur-core           45
3  resolve_token             spur-pm           36
4     sha256_hex       spur-worktree           24
5         labels            spur-tui           19
6        tempdir       spur-notebook           17
7            git            spur-cli           15
8            App       spur-notebook           13
9   load_artifact          spur-graph           10


**The trap, named:** `use std::fmt;` appears 121 times. There is also a workspace symbol named
`fmt` in `spur-mcp`. A name-only import-license would bind all 121 `std::fmt` imports — and any call
they "license" — to the workspace `fmt`. `path` (`std::path` vs a `spur-pm` `path`), `sleep`
(`tokio::time::sleep`), `render`, `init`, `run` are the same trap. Only **65** labels collide, but
they are the highest-frequency imports in the tree.

**The dependency this exposes:** import-aware recall (the 4,920) is **blocked on first making the
`imports` predicate itself crate-resolved** — i.e. distinguishing `use std::fmt` (external, no
license) from `use crate::render::fmt` (workspace, licenses a bind). You cannot use an unresolved
bare-name import as proof. **Import resolution is a Tier-1 prerequisite, not a sub-step.**


## 6. Frontier D — the method crater needs discrimination, not a hammer ⚠️

The 1,187 cross-crate `scope_match` method binds (§3) look like the singleton problem, but a blanket
`function_singleton_safe`-style crate guard would be **wrong** here — because some cross-crate method
calls are *genuine*. The label distribution proves both halves coexist:


In [ ]:
# Cross-crate scope_match method binds, by method name. Constructors dominate (collisions);
# domain verbs (update_issue/get_issue/create_issue) are genuine cross-crate API calls.
q("""
WITH m AS (
  SELECT dst.entity_name AS method,
         regexp_extract(src.file_path,'^crates/([^/]+)/',1) AS src_crate,
         regexp_extract(dst.file_path,'^crates/([^/]+)/',1) AS dst_crate
  FROM edges e
  JOIN nodes src ON src.stable_symbol_id=e.source_stable_id
  JOIN nodes dst ON dst.stable_symbol_id=e.target_stable_id
  WHERE e.relation='calls' AND e.bind_method='scope_match'
)
SELECT method, count(*) AS cross_crate_binds
FROM m WHERE src_crate<>'' AND dst_crate<>'' AND src_crate<>dst_crate
GROUP BY 1 ORDER BY cross_crate_binds DESC LIMIT 12
""")


           method  cross_crate_binds
0             new                620
1             now                222
2        embedded                 28
3         try_new                 27
4         default                 22
5            from                 21
6    update_issue                 18
7  active_validated                17
8    from_provider                 15
9       get_issue                 14
10   with_defaults                 12
11   from_parts_for_test          11


**`new` (620) and `now` (222) are almost entirely phantom** — `X::new()` in one crate
scope-matching `Y::new` in another because the enclosing-scope heuristic is receiver-type-blind.
But **`update_issue` / `get_issue` / `create_issue`** are real `spur-mcp → spur-pm` API calls. A
crate guard that dropped all 1,187 would erase those genuine edges.

**The remedy is *receiver-type discrimination*, not a crate gate:** resolve the receiver's type at
the call site (`X` in `X::new()`) and bind only to a method whose `enclosing_scope` is that type.
This is real type-resolution work — the first piece of Tier-1 that needs more than scope/name text.
A cheaper interim win: special-case the universal constructor names (`new`/`now`/`default`/`from`/
`try_new`) — collapsing those alone removes **~900 of the 1,187** phantoms at near-zero genuine cost.


## 7. The binding-precision ladder & sequencing

```mermaid
graph TD
    A["T1.b.2-a  singleton crate-safety<br/>calls v6 + references v7  ✅ DONE"]
    B["T1.b.1  same-crate recall recovery<br/>2,810 lone same-crate matches · no new evidence · SAFE"]
    C["T1.c.1  import predicate resolution<br/>crate-attribute use/import edges · PREREQUISITE"]
    D["T1.b.2-b  import-licensed cross-crate recall<br/>4,920 · needs C first"]
    E["T1.d.1  constructor-name method de-phantom<br/>~900 new/now/default · cheap"]
    F["T1.d.2  receiver-type method discrimination<br/>remaining ~287 · real type resolution"]

    A --> B
    A --> E
    C --> D
    B -.shares recall harness.-> D
    E --> F
    C -.same import index.-> F
```

**Recommended order (value ÷ risk):**
1. **T1.b.1 same-crate recall (2,810)** — safe, no new machinery, immediately lifts recall. Start here.
2. **T1.d.1 constructor de-phantom (~900)** — cheap precision win, no type resolution required.
3. **T1.c.1 import resolution** — the keystone; unblocks D and feeds F. The real design investment.
4. **T1.b.2-b import-licensed recall (4,920)** — gated on C.
5. **T1.d.2 receiver-type discrimination** — the deepest piece; do last, on top of C's import index.

Each step is an independent, golden-blessable, `RESOLVER_VERSION`-bumping change — the same
loop that landed v5/v6/v7.


## 8. The realization contract still binds (carry-over from Tier-0)

Every step above is **resolution semantics**, not schema/extractor/query. Tier-0's P0 lesson holds:
`current_manifest_version()` (`store/build.rs:135`) folds `RESOLVER_VERSION` into the manifest hash,
so each Tier-1 change **must bump `RESOLVER_VERSION`** (`build.rs:29`) or incremental builds keep
stale per-file binds and the fix never reaches the persisted graph. The guard
`manifest_version_changes_when_resolver_version_changes` enforces the discipline; the burden is
remembering to bump it on every resolver change — which v5→v6→v7 have each done.


## 9. Scope boundary — what is **not** Tier-1

- **Cross-artifact fusion** (resolving a call into a *dependency* crate's published artifact, or into
  std via a sysroot graph) is **Tier-2**. Tier-1 stays inside the worktree's closed world; it only
  decides *bind vs leave-unresolved*, never *invent an external node*.
- **Inference** (`called_by`, transitive reachability, impl→trait entailment) is **Tier-3**. Tier-1
  edges are all directly extracted, never derived.
- **Confidence-score calibration / governance** is **Tier-4**.
- Within Tier-1, **language coverage parity** (TS/Tsx HOF `references` currently TODO, `calls_dyn`
  cross-language) is a parallel track (T1.a) — orthogonal to the binding-precision ladder here.

**Tier-1 exit criterion:** for every *resolved* `calls`/`references` edge, the target is the symbol
the source actually named — verified by (a) zero cross-crate singleton/HOF binds, (b) cross-crate
binds licensed by a resolved import or a discriminated receiver type, (c) recall recovered for every
lone-workspace-def call that is provably in-scope. (a) is done; (b) and (c) are the work above.
